# Generation Metrics
### Faithfulness, Answer Relevancy, Answer Correctness, Answer Similarity — RAGAS

Corpus: `OWASP Top 10 for LLM Applications (2025)` — 10 named risk categories (LLM01–LLM10). Ground truth: 8 factual questions written directly from the document text, one per risk category, each with a single checkable reference answer.

## Step 1: Build the RAG pipeline

In [1]:
#!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf ragas openai python-dotenv -q

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

PDF_PATH = "OWASP-Top-10-for-LLMs-v2025.pdf"

pages = PyPDFLoader(PDF_PATH).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

rag_llm = ChatOllama(model="llama3.2:3b", temperature=0)

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

C:\Users\shiva\AppData\Local\Temp\ipykernel_20128\4103008943.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
incorrect startxref pointer(1)
parsing for Object Streams
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set


Loaded 45 pages -> 131 chunks -> 131 vectors


## Step 2: Ground truth — questions and reference answers
Eight factual questions, one per OWASP risk category, each with a reference answer written directly from the document.

In [3]:
ground_truth = [
    {
        "question": "What is Prompt Injection, according to LLM01?",
        "reference": "A Prompt Injection Vulnerability occurs when user prompts alter the LLM's behavior or output in unintended ways. The input does not need to be human-readable, only something the model parses.",
    },
    {
        "question": "What kinds of sensitive information can an LLM application expose, according to LLM02?",
        "reference": "Personal identifiable information (PII), financial details, health records, confidential business data, security credentials, legal documents, and proprietary training methods or source code.",
    },
    {
        "question": "What real attack against a model hosted on Hugging Face is cited as an example of Supply Chain risk under LLM03?",
        "reference": "PoisonGPT: an attacker bypassed Hugging Face's safety features by directly tampering with a model's parameters to spread misinformation.",
    },
    {
        "question": "What is Excessive Agency, according to LLM06?",
        "reference": "Excessive Agency is the vulnerability where an LLM-based system is granted excessive functionality, permissions, or autonomy to call functions or tools, letting it take unintended or damaging actions.",
    },
    {
        "question": "What does System Prompt Leakage warn about, according to LLM07?",
        "reference": "System prompts can inadvertently contain sensitive information, such as credentials or internal business rules, that was not meant to be discovered; the system prompt should never be treated as a secret or used as a security control.",
    },
    {
        "question": "What security risks affect vectors and embeddings in RAG systems, according to LLM08?",
        "reference": "Weaknesses in how vectors and embeddings are generated, stored, or retrieved can be exploited to inject harmful content, manipulate outputs, or leak sensitive data -- for example, in a multi-tenant vector database, one group's embeddings could be retrieved in response to another group's queries.",
    },
    {
        "question": "What is the main cause of misinformation in LLMs, according to LLM09?",
        "reference": "Hallucination -- the LLM generates content that seems accurate but is fabricated, filling gaps in its training data using statistical patterns without truly understanding the content.",
    },
    {
        "question": "What is Unbounded Consumption, according to LLM10?",
        "reference": "A risk where an LLM application allows excessive, uncontrolled inference operations, which can lead to denial of service, runaway costs, or model theft through extraction and cloning attacks.",
    },
]

import pandas as pd
pd.DataFrame(ground_truth)

,question,reference
0,"What is Prompt Injection, according to LLM01?",A Prompt Injection Vulnerability occurs when u...
1,What kinds of sensitive information can an LLM...,"Personal identifiable information (PII), finan..."
2,What real attack against a model hosted on Hug...,PoisonGPT: an attacker bypassed Hugging Face's...
3,"What is Excessive Agency, according to LLM06?",Excessive Agency is the vulnerability where an...
4,"What does System Prompt Leakage warn about, ac...",System prompts can inadvertently contain sensi...
5,What security risks affect vectors and embeddi...,Weaknesses in how vectors and embeddings are g...
6,What is the main cause of misinformation in LL...,Hallucination -- the LLM generates content tha...
7,"What is Unbounded Consumption, according to LL...",A risk where an LLM application allows excessi...


## Step 3: Retrieve + generate an answer for each question
Same retrieve-then-generate pattern as every other notebook this week — the only difference from here on is that we now *score* the result instead of just reading it.

In [4]:
RAG_PROMPT = """Answer the question using only the following context. Be concise.

Context:
{context}

Question: {question}
Answer:"""

for item in ground_truth:
    docs = vector_store.similarity_search(item["question"], k=3)
    item["contexts"] = [doc.page_content for doc in docs]
    context = "\n\n".join(item["contexts"])
    item["answer"] = rag_llm.invoke(
        RAG_PROMPT.format(context=context, question=item["question"])
    ).content.strip()

for item in ground_truth:
    print("Q:", item["question"])
    print("A:", item["answer"])
    print()

Q: What is Prompt Injection, according to LLM01?
A: Prompt Injection refers to the vulnerability in Large Language Models (LLMs) where an attacker can inject malicious or biased prompts to manipulate the model's output.

Q: What kinds of sensitive information can an LLM application expose, according to LLM02?
A: According to LLM02, an LLM application can expose:

1. Personal identifiable information (PII)
2. Proprietary algorithms or data
3. Confidential business data
4. Security credentials
5. Legal documents
6. Sensitive system architecture
7. API keys
8. Database credentials
9. User tokens

Q: What real attack against a model hosted on Hugging Face is cited as an example of Supply Chain risk under LLM03?
A: PoisonGPT.

Q: What is Excessive Agency, according to LLM06?
A: I don't have the specific information about "Excessive Agency" from LLM06. The provided context only mentions various types of prompt injection vulnerabilities and related topics, but does not explicitly define or di

## Step 4: Wire RAGAS to an LLM judge
RAGAS decomposes answers into atomic claims and checks each one — that needs a strong, consistent instruction-follower. A local 3B model is not reliable enough for this structured judging step, so the judge here is `gpt-4o-mini`, exactly like the reference pipeline on the slides ("LLM judge: gpt-4o-mini + embeddings"). Retrieval and generation above stay fully local on Ollama — only the *judge* is cloud-based.

One environment quirk first: the installed `ragas` release has a stale top-level import (`langchain_community.chat_models.vertexai`) for a provider this notebook never uses, and current `langchain-community` no longer ships that module. The cell below stubs it out before `ragas` is imported anywhere.

In [5]:
import sys, types

_stub = types.ModuleType("langchain_community.chat_models.vertexai")


class ChatVertexAI:
    pass


_stub.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = _stub

from dotenv import load_dotenv
load_dotenv()

from openai import AsyncOpenAI
from ragas.llms.base import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings

client = AsyncOpenAI()
judge_llm = llm_factory("gpt-4o-mini", client=client)
judge_embeddings = RagasOpenAIEmbeddings(client=client, model="text-embedding-3-small")

print("Judge ready.")

Judge ready.


## Step 5: Faithfulness — are the claims grounded in retrieved context?
RAGAS decomposes the answer into atomic statements and checks each one against the retrieved chunks. Score = supported statements / total statements.

In [6]:
from ragas.metrics.collections import Faithfulness

faithfulness_metric = Faithfulness(llm=judge_llm)

for item in ground_truth:
    result = await faithfulness_metric.ascore(
        user_input=item["question"],
        response=item["answer"],
        retrieved_contexts=item["contexts"],
    )
    item["faithfulness"] = result.value

pd.DataFrame(ground_truth)[["question", "faithfulness"]]

,question,faithfulness
0,"What is Prompt Injection, according to LLM01?",1.000000
1,What kinds of sensitive information can an LLM...,1.000000
2,What real attack against a model hosted on Hug...,0.500000
3,"What is Excessive Agency, according to LLM06?",0.666667
4,"What does System Prompt Leakage warn about, ac...",0.666667
5,What security risks affect vectors and embeddi...,0.400000
6,What is the main cause of misinformation in LL...,1.000000
7,"What is Unbounded Consumption, according to LL...",0.333333


## Step 6: Answer Relevancy — does the answer address the question?
RAGAS generates synthetic questions from the answer and compares their embeddings to the original question. An answer that fully addresses the question regenerates a similar question.

In [7]:
from ragas.metrics.collections import AnswerRelevancy

answer_relevancy_metric = AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings)

for item in ground_truth:
    result = await answer_relevancy_metric.ascore(
        user_input=item["question"],
        response=item["answer"],
    )
    item["answer_relevancy"] = result.value

pd.DataFrame(ground_truth)[["question", "answer_relevancy"]]

,question,answer_relevancy
0,"What is Prompt Injection, according to LLM01?",0.711049
1,What kinds of sensitive information can an LLM...,0.885055
2,What real attack against a model hosted on Hug...,0.367064
3,"What is Excessive Agency, according to LLM06?",0.000000
4,"What does System Prompt Leakage warn about, ac...",0.920848
5,What security risks affect vectors and embeddi...,0.990267
6,What is the main cause of misinformation in LL...,0.903912
7,"What is Unbounded Consumption, according to LL...",0.000000


## Step 7: Answer Correctness — is it factually right vs the reference?
Combines a claim-level F1 score (true/false positives and negatives against the reference) with semantic similarity, so it distinguishes hallucination (false positives) from incompleteness (false negatives).

In [8]:
from ragas.metrics.collections import AnswerCorrectness

answer_correctness_metric = AnswerCorrectness(llm=judge_llm, embeddings=judge_embeddings)

for item in ground_truth:
    result = await answer_correctness_metric.ascore(
        user_input=item["question"],
        response=item["answer"],
        reference=item["reference"],
    )
    item["answer_correctness"] = result.value

pd.DataFrame(ground_truth)[["question", "answer_correctness"]]

,question,answer_correctness
0,"What is Prompt Injection, according to LLM01?",0.458021
1,What kinds of sensitive information can an LLM...,0.540607
2,What real attack against a model hosted on Hug...,0.412984
3,"What is Excessive Agency, according to LLM06?",0.191018
4,"What does System Prompt Leakage warn about, ac...",0.593414
5,What security risks affect vectors and embeddi...,0.285104
6,What is the main cause of misinformation in LL...,0.421318
7,"What is Unbounded Consumption, according to LL...",0.114010


## Step 8: Answer Similarity — semantic closeness to the reference
Embeds both the generated answer and the reference answer, then scores their cosine similarity — a fast, bi-encoder check that the answer means the same thing as the reference, independent of exact wording.

In [9]:
from ragas.metrics.collections import SemanticSimilarity

answer_similarity_metric = SemanticSimilarity(embeddings=judge_embeddings)

for item in ground_truth:
    result = await answer_similarity_metric.ascore(
        reference=item["reference"],
        response=item["answer"],
    )
    item["answer_similarity"] = result.value

pd.DataFrame(ground_truth)[["question", "answer_similarity"]]

,question,answer_similarity
0,"What is Prompt Injection, according to LLM01?",0.832099
1,What kinds of sensitive information can an LLM...,0.662357
2,What real attack against a model hosted on Hug...,0.651935
3,"What is Excessive Agency, according to LLM06?",0.764070
4,"What does System Prompt Leakage warn about, ac...",0.573637
5,What security risks affect vectors and embeddi...,0.678879
6,What is the main cause of misinformation in LL...,0.685272
7,"What is Unbounded Consumption, according to LL...",0.456091


## Step 9: Scorecard

In [10]:
score_cols = ["faithfulness", "answer_relevancy", "answer_correctness", "answer_similarity"]
scorecard = pd.DataFrame(ground_truth)[["question"] + score_cols]
scorecard.loc["mean", "question"] = ""
scorecard.loc["mean", score_cols] = scorecard[score_cols].mean()
scorecard

,question,faithfulness,answer_relevancy,answer_correctness,answer_similarity
0,"What is Prompt Injection, according to LLM01?",1.000000,0.711049,0.458021,0.832099
1,What kinds of sensitive information can an LLM...,1.000000,0.885055,0.540607,0.662357
2,What real attack against a model hosted on Hug...,0.500000,0.367064,0.412984,0.651935
3,"What is Excessive Agency, according to LLM06?",0.666667,0.000000,0.191018,0.764070
4,"What does System Prompt Leakage warn about, ac...",0.666667,0.920848,0.593414,0.573637
5,What security risks affect vectors and embeddi...,0.400000,0.990267,0.285104,0.678879
6,What is the main cause of misinformation in LL...,1.000000,0.903912,0.421318,0.685272
7,"What is Unbounded Consumption, according to LL...",0.333333,0.000000,0.114010,0.456091
mean,,0.695833,0.597274,0.377059,0.663043


In [11]:
# Production targets from the slide deck's scorecard
targets = {
    "faithfulness": 0.85,
    "answer_relevancy": 0.80,
    "answer_correctness": 0.75,
    "answer_similarity": 0.75,
}

means = scorecard.loc["mean", score_cols]
for metric, target in targets.items():
    mean_score = means[metric]
    verdict = "PASS" if mean_score >= target else "BELOW MINIMUM"
    print(f"{metric:<20} mean={mean_score:.2f}  minimum={target:.2f}  -> {verdict}")

faithfulness         mean=0.70  minimum=0.85  -> BELOW MINIMUM
answer_relevancy     mean=0.60  minimum=0.80  -> BELOW MINIMUM
answer_correctness   mean=0.38  minimum=0.75  -> BELOW MINIMUM
answer_similarity    mean=0.66  minimum=0.75  -> BELOW MINIMUM


A few rows likely sit well below the production minimums above — that's the local `llama3.2:3b` + top-3 retrieval stack occasionally missing the right chunk or declining to answer, not a bug in the metrics. That gap between "the pipeline runs" and "the pipeline clears the bar" is exactly what these scores are for: they turn a silent quality regression into a number you can see, gate on, and go fix (better chunking, higher top-k, a stronger generation model).

## Try it yourself
1. Swap the judge to `gpt-4o` and see whether Faithfulness gets stricter or stays about the same.
2. Edit one `reference` answer to be subtly wrong and confirm Answer Correctness catches it while Faithfulness (which never looks at the reference) doesn't.
3. Add a 9th question the PDF can't answer at all and see how a model that should say "I don't know" scores on Faithfulness and Answer Relevancy instead.